# S2UT comparison with calibrated all-layer attention distillation

The comparison runs **Baseline → KL-based Attention Distillation → Annealed KL-based Attention Distillation → Attention-level Knowledge Distillation**. Learned T2TT cross-attention supervises the corresponding S2UT layers through the loss.

Match the Baseline's native fairseq model, auxiliary losses, optimizer, scheduler, logical batches, update budget, and checkpoint selection. Forward and generation use raw attention. Keep the Baseline and source data read-only. Work locally and publish verified ZIPs to Drive.

\[
\boxed{\mathcal L=\mathcal L_{\mathrm{S2UT}}+\lambda\mathcal L_{\mathrm{align}}}
\]


## GitHub access

For a private repository, create a fine-grained personal access token in GitHub **Settings → Developer settings → Personal access tokens → Fine-grained tokens**. Select the repository owner, choose **Only select repositories**, select `alignment-transfer-s2ut`, and grant **Contents → Read-only**. Set an expiration date. See the [GitHub token guide](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens).

In Colab, open **Secrets** (the key icon), add `GITHUB_TOKEN`, paste the token into its value, and enable **Notebook access**. The setup cell reads this secret for Git operations. Public repositories also work without a token. Successful clone/fetch and checkout in the setup cell confirm access to the requested revision.

To authenticate an additional repository, add its exact HTTPS URL to `GITHUB_AUTH_REPOS` in the settings cell and grant the same token read access to that repository. The benchmark is public and uses anonymous HTTPS by default.

After a runtime reset, use the same notebook revision, experiment settings, and Drive paths, then rerun the cells. To open the notebook after the repository becomes private, download the `.ipynb` while signed into GitHub and use **File → Upload notebook** in Colab, or open a copy saved in your Drive. Enable secret access for that notebook. The runtime token is used after the notebook opens.


## 1. Mount Drive and configure the run

Use a GPU runtime. If audio ZIPs are missing, first run a CPU runtime with `PREPARE_AUDIO_PACKS=True` through section 3.

`BENCHMARK` identifies the existing experiment, `PERSISTENT` the Drive destination, and `ROOT` the VM workspace. Give a new experiment its own `ROOT` and `PERSISTENT`. Keep the existing Baseline and corpus read-only.

### Kokoro JParaCrawl

Complete preprocessing and the 400k Baseline first. Set `CORPUS_MODE='kokoro'` and use the same `KOKORO_RUN_ID` here. Extract attention from the 200k-update SentencePiece T2TT trained on 300k JParaCrawl pairs for the 217,765 Kokoro pairs with matching IDs, splits, and text. [Kokoro Baseline notebook](kokoro_baseline.ipynb)

In [ ]:
#@title Process logging and progress
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def graceful_sigterm(function):
    """Turn a first POSIX SIGTERM into the ordinary interruption cleanup path."""
    from functools import wraps
    import os
    import signal
    import threading

    @wraps(function)
    def wrapped(*args, **kwargs):
        if (os.name != "posix" or threading.current_thread() is not threading.main_thread()
                or signal.getsignal(signal.SIGTERM) != signal.SIG_DFL):
            return function(*args, **kwargs)
        interrupted = False

        def interrupt(signum, frame):
            nonlocal interrupted
            if not interrupted:
                interrupted = True
                raise KeyboardInterrupt("SIGTERM")

        previous = signal.signal(signal.SIGTERM, interrupt)
        try:
            return function(*args, **kwargs)
        finally:
            signal.signal(signal.SIGTERM, previous)
    return wrapped


@graceful_sigterm
def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    termination_timeout = 75 if launch_env.get("AT_LOG_WANDB_DRAIN") == "1" else 5
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=termination_timeout)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


"""Standard-library GitHub authentication, embedded before Colab installation."""


GITHUB_GIT_WORKER = r'''
import base64
import os
import subprocess
import sys
from urllib.parse import quote

redactions = []
for i in range(int(os.environ.get("GIT_CONFIG_COUNT", "0"))):
    value = os.environ.get(f"GIT_CONFIG_VALUE_{i}", "")
    if value.startswith("Authorization: Basic "):
        encoded = value.split()[-1]
        token = base64.b64decode(encoded).decode("utf-8").split(":", 1)[1]
        redactions.extend((value, encoded, token, quote(token, safe="")))
redactions = sorted(set(redactions), key=len, reverse=True)
process = None
try:
    command = sys.argv[1:]
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    if command[position] in {"fetch", "pull"}:
        remote = next((arg for arg in command[position + 1:] if not arg.startswith("-")), "origin")
        if "://" not in remote:
            checked = subprocess.run([*command[:position], "remote", "get-url", "--all", remote],
                stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                text=True, encoding="utf-8", errors="replace")
            from urllib.parse import urlsplit
            for address in checked.stdout.splitlines():
                url = urlsplit(address)
                if url.username or url.password or url.query or url.fragment:
                    raise ValueError("unsafe stored remote")
    process = subprocess.Popen(command, stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding="utf-8", errors="replace", bufsize=1)
    for line in process.stdout:
        for secret in redactions:
            line = line.replace(secret, "[REDACTED]")
        print(line, end="", flush=True)
    code = process.wait()
except Exception:
    print("Git could not complete the requested operation.", file=sys.stderr, flush=True)
    code = 1
finally:
    if process is not None:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
if code:
    print("Check the repository URL and revision. For a private repository, add "
          "GITHUB_TOKEN to Colab Secrets, enable notebook access, and grant the "
          "fine-grained token Contents: Read-only for the selected repository. "
          "Check token expiry and any organization approval.", file=sys.stderr, flush=True)
sys.exit(code)
'''


def run_github_git(command, logs, *, runner, repositories, name="git", env=None, cwd=None,
                   capture_json=False):
    """Authenticate network Git calls with URL-scoped, process-only headers."""
    import base64
    import os
    from pathlib import Path
    import re
    import sys
    from urllib.parse import urlsplit

    command = list(map(str, command))
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    operation = command[position] if position < len(command) else ""
    if operation not in {"clone", "fetch", "pull", "ls-remote"}:
        return runner(command, logs, name=name, env=env, cwd=cwd, capture_json=capture_json)
    if Path(command[0]).name.lower() not in {"git", "git.exe"}:
        raise ValueError("GitHub authentication requires a git command")
    for argument in command:
        if "://" in argument:
            url = urlsplit(argument)
            if url.username or url.password or url.query or url.fragment:
                raise ValueError("Use repository URLs without credentials, queries, or fragments")
    scopes = []
    for repository in repositories:
        url = urlsplit(str(repository))
        if (url.scheme != "https" or url.netloc != "github.com" or url.query or url.fragment
                or not re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", url.path)):
            raise ValueError("Authentication scopes must be plain HTTPS GitHub repository URLs")
        scopes.append(url.geturl())

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass  # Public repositories remain accessible when Secrets are unavailable.
    if token is not None and (not isinstance(token, str) or any(ord(c) < 33 or ord(c) > 126 for c in token)):
        token = None
        raise ValueError("GITHUB_TOKEN must contain a valid GitHub token without whitespace")
    git_env = dict(os.environ if env is None else env)
    for key in list(git_env):
        if (key.startswith(("GIT_TRACE", "GIT_CONFIG_KEY_", "GIT_CONFIG_VALUE_"))
                or key in {"GIT_CONFIG_COUNT", "GIT_CONFIG_PARAMETERS", "GIT_CURL_VERBOSE", "GITHUB_TOKEN", "GH_TOKEN"}):
            git_env.pop(key)
    git_env.update(GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="Never", GIT_ASKPASS="", GIT_TRACE_REDACT="1")
    settings = [("credential.helper", ""), ("http.extraHeader", ""), ("http.followRedirects", "false")]
    header = None
    try:
        if token:
            header = "Authorization: Basic " + base64.b64encode(("x-access-token:" + token).encode()).decode()
            settings.extend(("http." + scope + ".extraHeader", header) for scope in dict.fromkeys(scopes))
        git_env["GIT_CONFIG_COUNT"] = str(len(settings))
        for i, (key, value) in enumerate(settings):
            git_env[f"GIT_CONFIG_KEY_{i}"] = key
            git_env[f"GIT_CONFIG_VALUE_{i}"] = value
        return runner([sys.executable, "-c", GITHUB_GIT_WORKER, *command], logs,
                      name=name, env=git_env, cwd=cwd, capture_json=capture_json)
    except Exception:
        raise RuntimeError("GitHub access failed. Check the logged URL/revision error and Colab Secrets "
                           "GITHUB_TOKEN access, selected repositories, Contents: Read-only, and expiry.") from None
    finally:
        token = header = None
        settings.clear()
        git_env.clear()


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None):
    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    if command and command[0] == 'git':
        repositories = [globals().get('REPO_URL', 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'),
                        *globals().get('GITHUB_AUTH_REPOS', [])]
        return run_github_git(command, LOGS, runner=run_process, repositories=repositories,
                              name=name, capture_json=capture_json, env=env, cwd=cwd)
    return run_process(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/benchmark_comparison/bootstrap')


In [ ]:
from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import json, os, subprocess, sys, time
from pathlib import Path


In [ ]:
#@title Corpus, Baseline, and storage
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = '5f87769a618296b44bb3949fb3f39f3b8af819fa'
BENCHMARK = '/content/drive/MyDrive/ja-en-direct-s2st-benchmark-data/training/experiment-001' #@param {type:"string"}
RUN_NAME = 's2ut-training-experiment-001-paper-fisher-v1' #@param {type:"string"}
PERSISTENT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/benchmark-match-drive-safe-001' #@param {type:"string"}
ROOT = '/content/alignment-transfer-work/benchmark-match-drive-safe-001' #@param {type:"string"}
CORPUS_MODE = 'benchmark' #@param ['benchmark', 'kokoro']
if CORPUS_MODE not in ('benchmark', 'kokoro'):
    raise ValueError('unknown CORPUS_MODE')
KOKORO = CORPUS_MODE == 'kokoro'
KOKORO_TEACHER_MANIFEST_OVERRIDE = '' #@param {type:"string"}
KOKORO_TEACHER_SNAPSHOT_OVERRIDE = '' #@param {type:"string"}

KOKORO_RUN_ID = 'kokoro-jparacrawl-l4-v1' #@param {type:"string"}
KOKORO_CORPUS_OVERRIDE = '' #@param {type:"string"}
if KOKORO:
    if Path(KOKORO_RUN_ID).name != KOKORO_RUN_ID or KOKORO_RUN_ID in ('', '.', '..'):
        raise ValueError('Set KOKORO_RUN_ID to a single directory name.')
    ROOT = str(Path('/content/alignment-transfer-work') / KOKORO_RUN_ID)
    PERSISTENT = str(Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / KOKORO_RUN_ID)

### Execution order and prerequisites

Install the environment, restore and register the Baseline, verify T2TT, extract all-layer attention and CTC, project the teacher, inspect top-p diagnostics, set `MASK_TOP_P`, calibrate λ and τ, train the three conditions, and evaluate them together.

Main training and T2TT extension default to `False`. Choose top-p explicitly and share one calibration artifact across the three transfer conditions. The loss-based comparison uses separate `research-v3-*` stages.


In [ ]:
#@title Training and evaluation switches
PREPARE_AUDIO_PACKS = False #@param {type:"boolean"}
ALL_LAYERS = True #@param {type:"boolean"}
RUN_DISTILLATION = False #@param {type:"boolean"}
GENERATE_WAVEFORMS = False #@param {type:"boolean"}
ASR_BLEU = False #@param {type:"boolean"}
T2TT_BASE_UPDATES = 20000 #@param {type:"integer"}
T2TT_TARGET_UPDATES = 200000 #@param {type:"integer"}
RUN_T2TT_EXTENSION = False #@param {type:"boolean"}
USE_EXTENDED_T2TT = True #@param {type:"boolean"}
T2TT_SNAPSHOT_UPDATES = 10000 #@param {type:"integer"}
T2TT_KEEP_LAST = 2 #@param {type:"integer"}
RUN_TEACHER_ARTIFACTS = False #@param {type:"boolean"}
T2TT_UPDATES = T2TT_BASE_UPDATES # compatibility: original 20k budget only
T2TT_TOKENIZER = 'character' #@param ['character', 'sentencepiece']
RUN_T2TT_SENTENCEPIECE = False #@param {type:"boolean"}
RUN_ANNEALED_DISTILLATION = False #@param {type:"boolean"}
RUN_MASK_DISTILLATION = False #@param {type:"boolean"}
RUN_ALIGNMENT_CALIBRATION = False #@param {type:"boolean"}
RUN_MASK_TOPP_DIAGNOSTICS = False #@param {type:"boolean"}
MASK_TOP_P = None # Set 0.8, 0.9, or 0.95 after reviewing the diagnostics.
CALIBRATION_BATCHES = 16 #@param {type:"integer"}
NEED_T2TT = RUN_TEACHER_ARTIFACTS or RUN_MASK_TOPP_DIAGNOSTICS or RUN_ALIGNMENT_CALIBRATION or RUN_DISTILLATION or RUN_ANNEALED_DISTILLATION or RUN_MASK_DISTILLATION
if not ALL_LAYERS:
    raise ValueError('Set ALL_LAYERS=True for the main comparison.')


In [ ]:
#@title Choose the T2TT run and its storage location
T2TT_BASE_ROOT = Path(ROOT) / 't2tt'
T2TT_EXTENDED_ROOT = Path(ROOT) / 't2tt-200k'
T2TT_ACTIVE_ROOT = T2TT_EXTENDED_ROOT if USE_EXTENDED_T2TT else T2TT_BASE_ROOT
T2TT_CHECKPOINT = T2TT_ACTIVE_ROOT / 'checkpoint_best.pt'
T2TT_ACTIVE_UPDATES = T2TT_TARGET_UPDATES if USE_EXTENDED_T2TT else T2TT_BASE_UPDATES
T2TT_ACTIVE_STAGE = 't2tt-200k' if USE_EXTENDED_T2TT else 't2tt'
T2TT_DEV_STAGE = 't2tt-dev-200k' if USE_EXTENDED_T2TT else 't2tt-dev'
T2TT_DEV_OUTPUT = Path(ROOT) / ('text-evaluation/t2tt-200k/dev-text.json' if USE_EXTENDED_T2TT else 'text-evaluation/dev-text.json')
T2TT_EXTERNAL_STAGE = 't2tt-flores-fleurs-200k' if USE_EXTENDED_T2TT else 't2tt-flores-fleurs'
T2TT_EXTERNAL_OUTPUT = Path(ROOT) / ('text-evaluation/t2tt-200k/flores-fleurs/flores-fleurs-text.json' if USE_EXTENDED_T2TT else 'text-evaluation/flores-fleurs/flores-fleurs-text.json')
T2TT_ATTENTION_STAGE = 't2tt-attention-200k' if USE_EXTENDED_T2TT else 'attention'
T2TT_ATTENTION_ROOT = Path(ROOT) / ('t2tt-attention-200k' if USE_EXTENDED_T2TT else 'mt')
CTC_STAGE = 'speech-text-alignment-200k' if USE_EXTENDED_T2TT else 'ctc'
CTC_ROOT = Path(ROOT) / ('speech-text-alignment-200k' if USE_EXTENDED_T2TT else 'ctc')
TEACHER_STAGE = 'teacher-alignment-200k' if USE_EXTENDED_T2TT else 'teacher'
TEACHER_ROOT = Path(ROOT) / ('teacher-alignment-200k' if USE_EXTENDED_T2TT else 'teacher')
TEACHER_AUDIT_STAGE = 'diagnostics-200k' if USE_EXTENDED_T2TT else 'diagnostics'
TEACHER_AUDIT_ROOT = Path(ROOT) / ('audit-200k' if USE_EXTENDED_T2TT else 'audit')


T2TT_TOKENIZER = globals().get('T2TT_TOKENIZER', 'character')
if T2TT_TOKENIZER not in ('character', 'sentencepiece'):
    raise ValueError('unknown T2TT tokenizer')
T2TT_IS_SPM = T2TT_TOKENIZER == 'sentencepiece'
T2TT_REQUIRES_COMPLETE = USE_EXTENDED_T2TT or T2TT_IS_SPM
T2TT_STATUS_MODULE = 'alignment_transfer_s2ut.spm_runtime' if T2TT_IS_SPM else 'alignment_transfer_s2ut.t2tt_extension'
if T2TT_IS_SPM:
    T2TT_ACTIVE_ROOT = Path(ROOT) / 't2tt-spm-200k'
    T2TT_ACTIVE_UPDATES = T2TT_TARGET_UPDATES
    T2TT_ACTIVE_STAGE = 't2tt-spm-200k'
    T2TT_DEV_STAGE = 't2tt-dev-spm-200k'
    T2TT_DEV_OUTPUT = Path(ROOT) / 'text-evaluation/t2tt-spm-200k/dev-text.json'
    T2TT_EXTERNAL_STAGE = 't2tt-flores-fleurs-spm-200k'
    T2TT_EXTERNAL_OUTPUT = Path(ROOT) / 'text-evaluation/t2tt-spm-200k/flores-fleurs/flores-fleurs-text.json'
    T2TT_ATTENTION_STAGE = 't2tt-attention-spm-200k'
    T2TT_ATTENTION_ROOT = Path(ROOT) / T2TT_ATTENTION_STAGE
    # CTC and units remain tokenizer-independent and may reuse verified artifacts.
    CTC_STAGE = 'speech-text-alignment-200k'
    CTC_ROOT = Path(ROOT) / CTC_STAGE
    TEACHER_STAGE = 'teacher-alignment-spm-200k'
    TEACHER_ROOT = Path(ROOT) / TEACHER_STAGE
    TEACHER_AUDIT_STAGE = 'diagnostics-spm-200k'
    TEACHER_AUDIT_ROOT = Path(ROOT) / 'audit-spm-200k'
T2TT_CHECKPOINT = T2TT_ACTIVE_ROOT / 'checkpoint_best.pt'

LAYER_FLAGS = ['--all-layers'] if ALL_LAYERS else []
if ALL_LAYERS:
    for stage_key,root_key in (('T2TT_ATTENTION_STAGE','T2TT_ATTENTION_ROOT'),('TEACHER_STAGE','TEACHER_ROOT'),('TEACHER_AUDIT_STAGE','TEACHER_AUDIT_ROOT')):
        globals()[stage_key] += '-all-layers'
        globals()[root_key] = globals()[root_key].with_name(globals()[root_key].name + '-all-layers')
if KOKORO:
    T2TT_ACTIVE_ROOT = Path(ROOT) / 'jparacrawl-sentencepiece-best'
    T2TT_CHECKPOINT = T2TT_ACTIVE_ROOT / 'checkpoint_best.pt'
    T2TT_ATTENTION_STAGE = 'kokoro-t2tt-attention-all-layers'
    T2TT_ATTENTION_ROOT = Path(ROOT) / T2TT_ATTENTION_STAGE
    CTC_STAGE = 'kokoro-speech-text-alignment'
    CTC_ROOT = Path(ROOT) / CTC_STAGE
    TEACHER_STAGE = 'kokoro-teacher-all-layers'
    TEACHER_ROOT = Path(ROOT) / TEACHER_STAGE
    TEACHER_AUDIT_STAGE = 'kokoro-teacher-diagnostics'
    TEACHER_AUDIT_ROOT = Path(ROOT) / TEACHER_AUDIT_STAGE
SCOPE_SUFFIX = ('-spm-200k' if T2TT_IS_SPM else '-200k' if USE_EXTENDED_T2TT else '') + ('-all-layers' if ALL_LAYERS else '')
if KOKORO:
    SCOPE_SUFFIX = '-kokoro-spm-all-layers'
RESEARCH_CONDITIONS = [
    ('distillation', 'KL-based Attention Distillation', RUN_DISTILLATION),
    ('annealed_distillation', 'Annealed KL-based Attention Distillation', RUN_ANNEALED_DISTILLATION),
    ('mask_distillation', 'Attention-level Knowledge Distillation', RUN_MASK_DISTILLATION)]
RESEARCH_ROOT = Path(ROOT) / ('research-v3' + SCOPE_SUFFIX)
TOPP_ROOT = RESEARCH_ROOT / 'top-p'
TOPP_REPORT = TOPP_ROOT / 'top-p.json'
CALIBRATION_ROOT = RESEARCH_ROOT / 'calibration'
CALIBRATION = CALIBRATION_ROOT / 'alignment-calibration.json'

In [ ]:
#@title Working paths and save interval
PYTHON = '/content/alignment-venv/bin/python'
REPO = Path('/content/alignment-transfer-s2ut')
LOGS = Path(ROOT) / 'execution'
PREPARED = Path(BENCHMARK) / 'data/s2ut/fairseq'
BACKUP = Path(BENCHMARK) / 'checkpoints' / RUN_NAME
RESTORED = Path(ROOT) / 'baseline-snapshot'
BASELINE = Path(ROOT) / 'registrations/baseline.json'
MANIFEST = Path(ROOT) / 'manifest/manifest.jsonl'
os.environ['HF_HOME'] = '/content/huggingface'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTHONUTF8'] = '1'
os.environ['MPLCONFIGDIR'] = '/content/matplotlib-cache'
assert not Path(ROOT).resolve().is_relative_to(Path(BENCHMARK).resolve())
assert not ASR_BLEU or GENERATE_WAVEFORMS

BENCHMARK_RUNTIME_REVISION = '4361e55a99c7f4ea3a00c5c138f6820c34a07d06'
BENCHMARK_RUNTIME = Path('/content/alignment-benchmark-runtime')
os.environ['S2ST_DRIVE_SAFE'] = '1'
os.environ['S2ST_BACKUP_MIN_SECONDS'] = '600'
os.environ['S2ST_BACKUP_MAX_GB'] = '1024'
assert not str(Path(ROOT).resolve()).startswith('/content/drive/')
assert not Path(ROOT).resolve().is_relative_to(Path(PERSISTENT).resolve())

if KOKORO:
    DATA = Path(ROOT) / 'data'
    COMMON = DATA / 'common'
    BENCHMARK_UNITS = DATA / 's2ut/units'
    PREPARED = DATA / 's2ut/fairseq'
    BACKUP = Path(PERSISTENT) / 'baseline'
    RESTORED = Path(ROOT) / 'baseline'

## 2. Set up and check the environment

Install Python 3.10.22, PyTorch and torchaudio 2.7.1, and pinned fairseq and benchmark revisions. Initial downloads and builds may take time.

Check dependencies, imports, and the native batch sampler before CPU tests and FP16 integration. Resolve failures and rerun the cell. External CTC, HuBERT, and vocoder require separate integration checks.

Each command logs startup, PID, live output, completion or failure, and the path under `LOGS`. After ten seconds of silence, a liveness notice reports that the process is still running.


In [ ]:
#@title Set up Python and fairseq
if not (REPO / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', REPO_URL, str(REPO), name='clone-alignment')
execute('git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION, name='fetch-alignment')
execute('git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION, name='checkout-alignment')
os.chdir(REPO)
execute(sys.executable, '-m', 'pip', 'install', 'uv', name='install-uv')
execute('uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv', name='create-python-environment')
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    execute(PYTHON, '-m', 'pip', 'install', *packages)


In [ ]:
#@title Install the pinned benchmark runtime
if not (BENCHMARK_RUNTIME / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', 'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git', str(BENCHMARK_RUNTIME))
execute('git', '-C', str(BENCHMARK_RUNTIME), 'fetch', 'origin', BENCHMARK_RUNTIME_REVISION, name='fetch-benchmark-runtime')
execute('git', '-C', str(BENCHMARK_RUNTIME), 'checkout', '--detach', BENCHMARK_RUNTIME_REVISION, name='checkout-benchmark-runtime')
execute(PYTHON, '-m', 'pip', 'install', 'PyYAML==6.0.2', '--no-deps', '-e', str(BENCHMARK_RUNTIME))
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')

if KOKORO:
    config = execute_json(PYTHON, '-c',
        "import json, yaml; print(json.dumps(yaml.safe_load(open('configs/data/colab.yaml', encoding='utf-8'))))",
        name='kokoro-paths')
    CORPUS_ROOT = Path(KOKORO_CORPUS_OVERRIDE or config['corpus_root'])
    SOURCE_MANIFEST = (CORPUS_ROOT / 'manifests/releases/accepted.jsonl'
                       if KOKORO_CORPUS_OVERRIDE else Path(config['accepted_manifest']))
    RELEASE_SUMMARY = SOURCE_MANIFEST.with_name('summary.json')
    if any(path.resolve().is_relative_to(CORPUS_ROOT.resolve()) for path in (Path(ROOT), Path(PERSISTENT))):
        raise ValueError('Choose an output directory outside the source corpus.')
    os.environ.pop('S2ST_INPUT_MAP', None)
    execute(PYTHON, '-m', 'pip', 'install', 'editdistance==0.8.1', 'librosa==0.10.2.post1',
        'accelerate==1.12.0', 'pandas==2.2.3', 'psutil==7.2.2', name='kokoro-runtime-dependencies')
    execute(PYTHON, '-m', 'pip', 'check', name='kokoro-dependency-check')

In [ ]:
def run(name, module, *arguments, gpu=False, env=None):
    arguments = list(map(str, arguments))
    payload = [PYTHON, '-m', module, *arguments]
    # Read-only environment checks and asset downloads have no stage output.
    if '--output' in arguments and module != 'alignment_transfer_s2ut.benchmark' or (
            module == 'alignment_transfer_s2ut.benchmark' and arguments[0] != 'plan'):
        if '--output' in arguments:
            output = Path(arguments[arguments.index('--output') + 1])
            work = output.parent if output.suffix in ('.json', '.jsonl', '.png') else output
            wrapper = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run',
                '--work', str(work), '--backup', str(Path(PERSISTENT) / 'stages' / name)]
            if module == 'alignment_transfer_s2ut.train_t2tt' or module == 'alignment_transfer_s2ut.benchmark' and arguments[0] == 'train':
                wrapper.append('--training')
            payload = [*wrapper, '--', *payload]
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    return execute(*command, '--', *payload, name='notebook:' + name, env=env)


In [ ]:
run('environment', 'alignment_transfer_s2ut.environment', '--native', '--benchmark',
    *([] if PREPARE_AUDIO_PACKS else ['--cuda']))
if not PREPARE_AUDIO_PACKS:
    runtime = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'runtime', name='runtime-details')
    (Path(ROOT) / 'runtime.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
    print(runtime)
run('cpu-tests', 'pytest', '-v', '--tb=short', '-m', 'not gpu')


### Native CUDA integration

Use synthetic data to check official Trainer FP16 updates, FP32 priors, and single and batched T2TT attention parity. CPU audio ZIP preparation skips this step.

Check two updates for each all-layer transfer condition, saving after the first and restoring for the second. `tiny_all_layer_gpu_preflight` displays and records peak VRAM for the two-layer test fixture.


In [ ]:
if not PREPARE_AUDIO_PACKS:
    execute(PYTHON, '-m', 'pytest', 'tests/test_benchmark_native.py::test_native_fp16_trainer_and_fp32_prior',
        'tests/test_all_layers_native.py::test_all_layer_fp16_trainer_and_fp32_prior',
        'tests/test_all_layers.py::test_all_layer_extraction_on_cuda',
        'tests/test_attention_extraction.py::test_batched_native_attention_matches_single_pair_on_cuda',
        '-v', '-s', '--tb=short', '--basetemp', '/content/native-fp16-test', cwd=REPO, name='native-fp16-test')
else:
    log_event('native-fp16-test', 'skipped', LOGS, 'CPU audio preparation')


## 3. Prepare inputs

Extract benchmark WAV ZIPs and prepared metadata to the VM, preserving their logical paths and training data.

- With completed ZIPs, set `PREPARE_AUDIO_PACKS=False`.
- To create missing ZIPs, set `PREPARE_AUDIO_PACKS=True` on CPU. After completion, switch to GPU, set it to `False`, and rerun from the top.

This stage transfers, verifies, and extracts input. Watch `checked` and `zip` for progress and use `status=running` for liveness. The logger displays the path under `ROOT/execution/*.log`. An interruption terminates the child processes and retains completed local packs. [Colab storage and recovery](https://github.com/Yaaamashiro/alignment-transfer-s2ut/blob/main/docs/COLAB_RUNTIME.md)

For Kokoro, restore the WAV ZIPs, units, and fairseq data from its preparation notebook.

In [ ]:
def stage_inputs_live(command):
    return execute_json(*command, name='stage-inputs')

def kokoro_stage(name, module, *arguments, work, restore_only=False):
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(work),
        '--backup', str(Path(PERSISTENT) / 'stages' / name)]
    if restore_only:
        command.append('--restore-only')
    execute(*command, '--', PYTHON, '-m', module, *map(str, arguments), name=name)

def kokoro_common(restore_only=False):
    os.environ['CORPUS_ROOT'] = str(CORPUS_ROOT)
    kokoro_stage('kokoro-common', 'alignment_transfer_s2ut.benchmark_data', 'kokoro',
        '--source', SOURCE_MANIFEST, '--summary', RELEASE_SUMMARY, '--corpus-root', CORPUS_ROOT,
        '--output', COMMON, '--resume', work=COMMON, restore_only=restore_only)
    lock = json.loads((COMMON / 'dataset-lock.json').read_text(encoding='utf-8'))
    if lock['total_pairs'] != config['kokoro_expected_pairs']:
        raise ValueError(f"Kokoro train/dev count {lock['total_pairs']} differs from the configured release")
    print({'pairs': lock['total_pairs'], 'splits': lock['splits'],
        'estimated_waveform_GiB': round(lock['wav_bytes_estimate'] / 1024**3, 1)})

def kokoro_stage_audio(create=False):
    staged = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'stage-kokoro',
        '--common', str(COMMON), '--corpus-root', str(CORPUS_ROOT), '--packs', str(Path(PERSISTENT) / 'wav-packs'),
        '--local', str(Path(ROOT) / 'staged-inputs'), *(['--create-audio'] if create else []), name='kokoro-audio')
    if not create:
        os.environ['S2ST_INPUT_MAP'] = staged['input_map']
    print(staged)

def kokoro_prepare(restore_only=False):
    kokoro_stage('kokoro-benchmark-units', 'alignment_transfer_s2ut.kokoro_runtime', 'units',
        '--common', COMMON, '--output', BENCHMARK_UNITS, '--cache', '/content/alignment-assets',
        work=BENCHMARK_UNITS, restore_only=restore_only)
    kokoro_stage('kokoro-fairseq', 'alignment_transfer_s2ut.kokoro_runtime', 'prepare',
        '--common', COMMON, '--units', BENCHMARK_UNITS, '--output', PREPARED,
        work=PREPARED, restore_only=restore_only)

def kokoro_baseline(restore_only=False):
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.kokoro_runtime', 'baseline',
        '--data', str(DATA), '--work', str(RESTORED), '--backup', str(BACKUP),
        *(['--restore-only'] if restore_only else ['--train']), name='kokoro-baseline')

if KOKORO:
    if PREPARE_AUDIO_PACKS:
        raise ValueError('Prepare the Kokoro WAV ZIPs in kokoro_baseline.ipynb.')
    kokoro_common(restore_only=True)
    kokoro_stage_audio()
    kokoro_prepare(restore_only=True)
else:
    stage_command = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'stage-inputs',
        '--benchmark', BENCHMARK, '--local', '/content/alignment-transfer-inputs/benchmark-match-drive-safe-001']
    if PREPARE_AUDIO_PACKS:
        staged = stage_inputs_live([*stage_command, '--create-audio'])
        raise RuntimeError('WAV ZIPs are ready. Switch to a GPU runtime, set PREPARE_AUDIO_PACKS=False, and rerun from the top.')
    staged = stage_inputs_live(stage_command)
    os.environ['S2ST_INPUT_MAP'] = staged['input_map']
    print(staged)

## 4. Restore and register the Baseline

On the first run, verify the latest completed snapshot and restore it to a separate location. Later runs use the selected snapshot.

Check `checkpoint_updates` for the best checkpoint step, `training_budget` for completed updates, and `selection_policy`. Match the transfer budget to `training_budget`.

For Kokoro, restore the dev-selected best checkpoint from the completed 400,000-update Baseline.

In [ ]:
if KOKORO:
    kokoro_baseline(restore_only=True)
else:
    run('restore-baseline', 'alignment_transfer_s2ut.benchmark', 'restore', '--backup', BACKUP, '--output', RESTORED, '--resume')
run('register-baseline', 'alignment_transfer_s2ut.benchmark', 'register',
    '--checkpoint', RESTORED / 'checkpoints/checkpoint_best.pt', '--data', PREPARED,
    '--recipe-lock', RESTORED / 's2ut-recipe-lock.json', '--metadata', RESTORED / 'research-metadata.json',
    '--output', BASELINE, '--resume')
baseline = json.loads(BASELINE.read_text())
print({key: baseline[key] for key in ['checkpoint_updates', 'training_budget', 'checkpoint_selection', 'selection_policy']})

BACKBONE_CONFIG = Path(ROOT) / 'backbone.json'
BACKBONE_CONFIG.write_text(json.dumps({'backbone': baseline['backbone']}), encoding='utf-8')

## 5. Import shared data and units

Use the Baseline's full train and dev set. Check saved HuBERT units against prepared targets. Teacher construction and model selection use train or dev. Kokoro uses the same units as its Baseline.

In [ ]:
run('full-manifest', 'alignment_transfer_s2ut.benchmark_data', 'manifest',
    '--common', COMMON if KOKORO else Path(BENCHMARK) / 'data/common', '--prepared', PREPARED, '--output', MANIFEST, '--resume')
run('import-units', 'alignment_transfer_s2ut.benchmark_data', 'units', '--manifest', MANIFEST,
    '--benchmark-units', BENCHMARK_UNITS if KOKORO else Path(BENCHMARK) / 'data/s2ut/units', '--prepared', PREPARED,
    '--output', Path(ROOT) / 'units', '--resume')

## 6. Restore and evaluate T2TT

For Kokoro, proceed to 6.2 to verify the trained SentencePiece model and original text.

### 6.1 Existing 20k T2TT

Use the 20k run as the read-only source for a separate 200k continuation. `USE_EXTENDED_T2TT` selects the same model for downstream evaluation, attention, and teacher construction. Section 6.1a seeds the new stage from a verified Drive snapshot.

In [ ]:
if KOKORO:
    log_event('legacy-t2tt', 'skipped', LOGS, 'Kokoro uses the fixed JParaCrawl teacher')
else:
    # 20k source is read-only when extending; only 6.1a creates the new run.
    if T2TT_IS_SPM or RUN_T2TT_EXTENSION or USE_EXTENDED_T2TT:
        log_event('t2tt-base', 'skipped', LOGS, 'verified 20k source is read-only; continue in 6.1a')
    elif NEED_T2TT:
        run('t2tt', 'alignment_transfer_s2ut.train_t2tt', '--manifest', MANIFEST,
            '--config', BACKBONE_CONFIG, '--output', T2TT_BASE_ROOT, '--max-updates', T2TT_BASE_UPDATES,
            '--device', 'cuda', '--loader-workers', 2, '--validation-interval', 1000,
            '--save-interval-updates', 100, '--resume-if-exists', gpu=True)
    else:
        log_event('t2tt-base', 'skipped', LOGS, 'teacher stage not requested')

### 6.1a Extend verified T2TT from 20k to 200k

Prepare the environment and original inputs in sections 1–5, run extension here, then run dev and external evaluation in 6.1b and 6.1c. Review curves and translations before setting `RUN_TEACHER_ARTIFACTS=True` for 6.2.

Write to `ROOT/t2tt-200k` and `PERSISTENT/stages/t2tt-200k`. Initialize continuation from verified 20k last with its optimizer, RNG, cursor, best, and history. Resume subsequent runs from the 200k stage. The source hash selects the original trainer code and preserves the checkpoint contract.

Validate every 1000 updates and save locally every 100. Publish verified full Drive ZIPs every 10000 updates by default, at milestones, and at completion. Keep the latest two and milestones after a verified upload. [Extension workflow](../../docs/T2TT_EXTENSION.md)

`[T2TT loss]` reports train loss, the latest dev loss and measured update, and LR every 100 updates or on new dev results. The monitor polls local history once per second. Drive snapshots include `metrics-summary.json` with up to 2000 points. Live output can be ahead of durable saves.

For an active run, download `ROOT/t2tt-200k/metrics.jsonl` from Colab's Files panel. The updated monitor takes effect at the next normal resume, using the original trainer, controller, and checkpoint. Other code cells wait for the active cell to finish.


In [ ]:
if KOKORO:
    log_event('legacy-t2tt', 'skipped', LOGS, 'Kokoro uses the fixed JParaCrawl teacher')
else:
    if not T2TT_IS_SPM and RUN_T2TT_EXTENSION and (NEED_T2TT):
        execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', 't2tt-200k',
            '--logs', str(LOGS), '--monitor-gpu', '--', PYTHON, '-m', 'alignment_transfer_s2ut.t2tt_extension', 'extend',
            '--source-work', str(T2TT_BASE_ROOT), '--source-backup', str(Path(PERSISTENT) / 'stages/t2tt'),
            '--work', str(T2TT_EXTENDED_ROOT), '--backup', str(Path(PERSISTENT) / 'stages/t2tt-200k'),
            '--manifest', str(MANIFEST), '--config', str(BACKBONE_CONFIG), '--source-updates', str(T2TT_BASE_UPDATES),
            '--target-updates', str(T2TT_TARGET_UPDATES), '--snapshot-updates', str(T2TT_SNAPSHOT_UPDATES),
            '--keep-last', str(T2TT_KEEP_LAST), name='notebook:t2tt-200k')
    else:
        log_event('t2tt-200k', 'skipped', LOGS, 'extension disabled or teacher stage not requested')

### 6.1a-S Train SentencePiece Unigram from 0 to 200k

Set `T2TT_TOKENIZER='sentencepiece'` and rerun the storage-path cell. This selects a new `t2tt-spm-200k` stage. Set `RUN_T2TT_SENTENCEPIECE=False` to evaluate a completed model.

Fit separate Japanese and English Unigram tokenizers on train only with 8000 target pieces each, identity normalization, coverage 1.0, fixed order, one thread, whitespace preservation, and no dummy prefix. Dev unknown characters are reported as UNK.

Match the character run's Transformer, optimizer, LR, batch 8×4, 20000 max tokens, 200k updates, validation interval 1000, and local save interval 100. Use verified Drive ZIPs every 10k updates, retain the latest two and milestones, and share the loss logging and summary format.

Run 6.1b, 6.1c, and 6.1d after completion. SentencePiece dev and fixed-300 evaluation, attention, teacher, and S2UT use distinct `spm-200k` destinations. Reuse CTC and units when input and config hashes match. Vocabulary counts and sequence lengths are in `t2tt-spm-200k/tokenization.json`. [SentencePiece specification](../../docs/SENTENCEPIECE.md)


In [ ]:
if KOKORO:
    log_event('legacy-t2tt', 'skipped', LOGS, 'Kokoro uses the fixed JParaCrawl teacher')
else:
    if T2TT_IS_SPM and RUN_T2TT_SENTENCEPIECE and (NEED_T2TT):
        execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', T2TT_ACTIVE_STAGE,
            '--logs', str(LOGS), '--monitor-gpu', '--', PYTHON, '-m', 'alignment_transfer_s2ut.spm_runtime', 'train',
            '--work', str(T2TT_ACTIVE_ROOT), '--backup', str(Path(PERSISTENT) / 'stages' / T2TT_ACTIVE_STAGE),
            '--snapshot-updates', str(T2TT_SNAPSHOT_UPDATES), '--keep-last', str(T2TT_KEEP_LAST), '--',
            PYTHON, '-m', 'alignment_transfer_s2ut.train_t2tt', '--manifest', str(MANIFEST), '--config', str(BACKBONE_CONFIG),
            '--output', str(T2TT_ACTIVE_ROOT), '--tokenizer', 'sentencepiece', '--source-vocab-size', '8000', '--target-vocab-size', '8000',
            '--sentencepiece-model-type', 'unigram', '--max-updates', str(T2TT_TARGET_UPDATES), '--batch-size', '8',
            '--update-frequency', '4', '--max-tokens', '20000', '--loader-workers', '2', '--validation-interval', '1000',
            '--save-interval-updates', '100', '--device', 'cuda', '--resume-if-exists', name='notebook:t2tt-spm-200k')
    else:
        log_event('t2tt-spm-200k', 'skipped', LOGS, 'SentencePiece training disabled or character/teacher stage not requested')

### 6.1b T2TT in-domain dev evaluation

Evaluate the selected best checkpoint. The 200k run uses `t2tt-dev-200k` and `text-evaluation/t2tt-200k/` separately from 20k results.


In [ ]:
if KOKORO:
    log_event('legacy-t2tt', 'skipped', LOGS, 'Kokoro uses the fixed JParaCrawl teacher')
else:
    if NEED_T2TT:
        execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'restore-inputs',
            '--work', str(T2TT_ACTIVE_ROOT), '--backup', str(Path(PERSISTENT) / 'stages' / T2TT_ACTIVE_STAGE),
            '--expected-updates', str(T2TT_ACTIVE_UPDATES), name='restore-active-t2tt')
        if T2TT_REQUIRES_COMPLETE:
            print(execute_json(PYTHON, '-m', T2TT_STATUS_MODULE, 'status',
                '--work', str(T2TT_ACTIVE_ROOT), '--target-updates', str(T2TT_TARGET_UPDATES), name='verify-extension-complete'))
        run(T2TT_DEV_STAGE, 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', MANIFEST,
            '--checkpoint', T2TT_CHECKPOINT, '--output', T2TT_DEV_OUTPUT, '--device', 'cuda', '--resume', gpu=True)
        print(json.loads(T2TT_DEV_OUTPUT.read_text(encoding='utf-8')))
    else:
        log_event('t2tt-dev', 'skipped', LOGS, 'teacher stage not requested')

### 6.1c Optional FLEURS/FLORES external text BLEU

Set `RUN_EXTERNAL_TEXT=True` to evaluate the selected `checkpoint_best.pt` on the same saved 8C set of 300 samples. Match dev beam, output limit, and length penalty, and join English references after generation.

Use saved results under `BENCHMARK/diagnostics/flores-fleurs-cascade/`. A missing set stops evaluation. If several exist, set `SOURCE_8C` to the saved identity directory.

A fresh VM verifies and restores the best checkpoint from the completed ZIP under `PERSISTENT/stages/<T2TT_ACTIVE_STAGE>`. Restore saved dev results when available. If dev is absent, evaluate the external set with fixed settings and omit the in-domain comparison. Verify checkpoint, generation settings, and predictions before displaying an available dev comparison.

Native BLEU supports comparison with dev. Benchmark-normalized BLEU scores the text against the shared references used by prior Cascade and S2UT evaluations. Those systems' final ASR-BLEU also reflects speech generation and ASR. Results and partial predictions use verified ZIPs in `T2TT_EXTERNAL_STAGE`. [Details and recovery](../../docs/EXTERNAL_TEXT_EVALUATION.md)

Use the selected `T2TT_CHECKPOINT`. The 200k stage writes to `t2tt-flores-fleurs-200k` and shares the fixed 8C input with 20k.


In [ ]:
if KOKORO:
    log_event('legacy-t2tt', 'skipped', LOGS, 'Kokoro uses the fixed JParaCrawl teacher')
else:
    #@title Evaluate T2TT on the 300 saved 8C pairs
    RUN_EXTERNAL_TEXT = False #@param {type:"boolean"}
    SOURCE_8C = '' #@param {type:"string"}
    if RUN_EXTERNAL_TEXT:
        import random
        import pandas as pd
        from IPython.display import display
        fixed_set = Path(ROOT) / 'external-text-input/fixed-8c.json'
        external_output = T2TT_EXTERNAL_OUTPUT
        dev_output = T2TT_DEV_OUTPUT
        execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'restore-inputs',
            '--work', str(T2TT_ACTIVE_ROOT), '--backup', str(Path(PERSISTENT) / 'stages' / T2TT_ACTIVE_STAGE),
            '--expected-updates', str(T2TT_ACTIVE_UPDATES), '--dev-backup', str(Path(PERSISTENT) / 'stages' / T2TT_DEV_STAGE),
            '--dev-output', str(dev_output), name='restore-t2tt-for-external')
        if T2TT_REQUIRES_COMPLETE:
            execute_json(PYTHON, '-m', T2TT_STATUS_MODULE, 'status',
                '--work', str(T2TT_ACTIVE_ROOT), '--target-updates', str(T2TT_TARGET_UPDATES), name='verify-extension-before-external')
        execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'prepare',
            '--benchmark', BENCHMARK, '--output', str(fixed_set),
            *(['--source-8c', SOURCE_8C] if SOURCE_8C else []), name='freeze-saved-8c-text')
        run(T2TT_EXTERNAL_STAGE, 'alignment_transfer_s2ut.evaluate_t2tt_external', 'evaluate',
            '--fixed-set', fixed_set, '--checkpoint', T2TT_CHECKPOINT,
            '--dev-evaluation', dev_output, '--output', external_output, '--device', 'cuda', '--resume', gpu=True)
        # Verify even when the ZIP runtime skips an already completed stage.
        result = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'verify',
            '--fixed-set', str(fixed_set), '--checkpoint', str(T2TT_CHECKPOINT),
            '--dev-evaluation', str(dev_output), '--output', str(external_output), name='verify-external-text')
        print('T2TT external FLEURS/FLORES')
        if result.get('source_length_bleu'):
            display(pd.DataFrame(result['source_length_bleu']))
        if result.get('generation_diagnostics'):
            display(pd.DataFrame([result['generation_diagnostics']]))
        for label, value in [
            ('samples', result['samples']), ('checkpoint (best step)', result['checkpoint_step']),
            ('native text BLEU', f"{result['text_bleu']:.4f}"),
            ('benchmark normalized BLEU', f"{result['benchmark_normalized_bleu']:.4f}"),
            ('EOS reached', f"{result['eos_reached_ratio']:.4f}"),
            ('UNK char rate', f"{result['source_unk_rate']:.6f}"),
            ('samples with UNK', f"{result['samples_with_unk']} / {result['samples']}")]:
            print(f'{label:26}: {value}')
        comparison = []
        if dev_output.is_file():
            dev = json.loads(dev_output.read_text(encoding='utf-8'))
            comparison.append({'Dataset': 'In-domain dev', 'native text BLEU': dev['text_bleu'],
                'benchmark normalized BLEU': None})
        else:
            log_event('in-domain-dev-comparison', 'skipped', LOGS, 'dev-text.json is absent; external evaluation uses existing fixed defaults')
        comparison.append({'Dataset': 'External FLEURS/FLORES', 'native text BLEU': result['text_bleu'],
            'benchmark normalized BLEU': result['benchmark_normalized_bleu']})
        display(pd.DataFrame(comparison))
        predictions = [json.loads(line) for line in external_output.with_name(
            external_output.stem + '-predictions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
        examples = random.Random(20261027).sample(predictions, min(10, len(predictions)))
        display(pd.DataFrame([{'sample_id': r['sample_id'], 'JA': r['source_text'],
            'Reference EN': r['reference'], 'T2TT EN': r['hypothesis'],
            'sentence BLEU': r['sentence_bleu']} for r in examples]))
    else:
        log_event(T2TT_EXTERNAL_STAGE, 'skipped', LOGS, 'RUN_EXTERNAL_TEXT=False')

### 6.1d Review learning curves and translations

Review dev and external BLEU, ten translations, and the curves and comparison below. Missing historical 20k values appear blank. Set `RUN_TEACHER_ARTIFACTS=True` after review to proceed to 6.2. The default Run all stops at the 6.2 gate.


In [ ]:
if KOKORO:
    log_event('legacy-t2tt', 'skipped', LOGS, 'Kokoro uses the fixed JParaCrawl teacher')
else:
    #@title Learning curves and saved 20k / 200k-best results
    if not NEED_T2TT:
        log_event('t2tt-learning-review', 'skipped', LOGS, 'teacher stage not requested')
    else:
        from IPython.display import Image, display
        if T2TT_REQUIRES_COMPLETE:
            execute(PYTHON, '-m', T2TT_STATUS_MODULE, 'report',
                '--work', str(T2TT_ACTIVE_ROOT), '--output', str(T2TT_ACTIVE_ROOT / 'diagnostics'), name='extension-learning-curve')
            display(Image(filename=str(T2TT_ACTIVE_ROOT / 'diagnostics/learning-curve.png')))
        import pandas as pd
        comparison = []
        for label, folder, dev_path, external_path in [
            ('20k', T2TT_BASE_ROOT, Path(ROOT) / 'text-evaluation/dev-text.json',
             Path(ROOT) / 'text-evaluation/flores-fleurs/flores-fleurs-text.json'),
            ('200k-best', T2TT_EXTENDED_ROOT, Path(ROOT) / 'text-evaluation/t2tt-200k/dev-text.json',
             Path(ROOT) / 'text-evaluation/t2tt-200k/flores-fleurs/flores-fleurs-text.json'),
            ('SentencePiece-200k-best', Path(ROOT) / 't2tt-spm-200k', Path(ROOT) / 'text-evaluation/t2tt-spm-200k/dev-text.json',
             Path(ROOT) / 'text-evaluation/t2tt-spm-200k/flores-fleurs/flores-fleurs-text.json')]:
            values = {'Condition': label}
            if dev_path.is_file():
                saved = json.loads(dev_path.read_text(encoding='utf-8'))
                values.update({'Dev BLEU': saved['text_bleu'], 'best checkpoint step': saved['checkpoint_step']})
            if external_path.is_file():
                saved = json.loads(external_path.read_text(encoding='utf-8'))
                values.update({'FLEURS native BLEU': saved['text_bleu'],
                    'FLEURS normalized BLEU': saved['benchmark_normalized_bleu'], 'best checkpoint step': saved['checkpoint_step']})
            comparison.append(values)
        display(pd.DataFrame(comparison))

### 6.2 Extract attention and align CTC

Extract T2TT attention in batches of up to eight pairs, then align Japanese and English CTC. Hash the checkpoint once at startup. Progress reports new and reused counts, pairs per second, and estimated time remaining.

Extract FP32 teacher-forced attention and reuse verified packets with `--resume`. Extraction batch settings are independent of training. With `ALL_LAYERS=True`, capture all layers in one forward pass and match layer `l` to S2UT layer `l`. `max_attention_cells` limits B×L×T×S. The `-all-layers` paths separate these artifacts from single-layer outputs. [Comparison workflow](../../docs/BENCHMARK_COMPARISON.md)

For Kokoro, first verify T2TT source train and dev hashes and every corresponding ID, split, and text.

In [ ]:
if KOKORO:
    if NEED_T2TT:
        teacher_stages = (T2TT_ATTENTION_STAGE, CTC_STAGE, TEACHER_STAGE)
        saved = all(any((Path(PERSISTENT) / 'stages' / stage).glob('*/snapshot.json')) for stage in teacher_stages)
        if not RUN_TEACHER_ARTIFACTS and not saved:
            raise ValueError('Set RUN_TEACHER_ARTIFACTS=True to build the Kokoro teacher for the first time.')
        source_manifest = KOKORO_TEACHER_MANIFEST_OVERRIDE or config['teacher_source_manifest']
        snapshot = KOKORO_TEACHER_SNAPSHOT_OVERRIDE or config['teacher_snapshot']
        text_manifest = Path(ROOT) / 'teacher-text/manifest.jsonl'
        execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_manifest',
            '--input', source_manifest, '--output', str(text_manifest), '--resume', name='kokoro-teacher-text')
        checked = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.kokoro_runtime', 'restore-teacher',
            '--snapshot', snapshot, '--output', str(T2TT_ACTIVE_ROOT), '--updates', str(config['teacher_updates']),
            '--source-manifest', str(text_manifest), '--speech-manifest', str(MANIFEST),
            '--backbone', str(BACKBONE_CONFIG), name='verify-kokoro-teacher')
        print(checked)
else:
    teacher_backups = [Path(PERSISTENT) / 'stages' / stage for stage in (T2TT_ATTENTION_STAGE, CTC_STAGE, TEACHER_STAGE)]
    CAN_RESTORE_TEACHER = all(any(path.glob('*/snapshot.json')) for path in teacher_backups)
    if T2TT_REQUIRES_COMPLETE and (NEED_T2TT) and not RUN_TEACHER_ARTIFACTS and not CAN_RESTORE_TEACHER:
        raise RuntimeError('Review BLEU, translation examples, and learning curves in sections 6.1b through 6.1d. Then set RUN_TEACHER_ARTIFACTS=True and run section 6.2.')
    if T2TT_REQUIRES_COMPLETE and (NEED_T2TT):
        checked = execute_json(PYTHON, '-m', T2TT_STATUS_MODULE, 'status',
            '--work', str(T2TT_ACTIVE_ROOT), '--target-updates', str(T2TT_TARGET_UPDATES), name='verify-teacher-before-attention')
        for required_metrics in (T2TT_DEV_OUTPUT,):
            if not required_metrics.is_file() or json.loads(required_metrics.read_text(encoding='utf-8')).get('checkpoint_hash') != checked['checkpoint_hash']:
                raise RuntimeError('Complete dev evaluation for the selected 200k best checkpoint in section 6.1b before running section 6.2.')

if NEED_T2TT:
    teacher_source_args = ['--teacher-training-manifest', text_manifest] if KOKORO else []
    run(T2TT_ATTENTION_STAGE, 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', MANIFEST,
        '--checkpoint', T2TT_CHECKPOINT, '--output', T2TT_ATTENTION_ROOT, '--device', 'cuda', '--resume', *LAYER_FLAGS, *teacher_source_args, gpu=True)
    run(CTC_STAGE, 'alignment_transfer_s2ut.align_speech_text', '--manifest', MANIFEST,
        '--config', 'configs/alignment/ctc.yaml', '--output', CTC_ROOT, '--device', 'cuda', '--resume', gpu=True)
else:
    log_event('teacher-preparation', 'skipped', LOGS, 'teacher stage not requested')

### 6.3 Project onto speech time and diagnose alignment

Project attention into S2UT time coordinates. Review coverage, invalid-row rate, entropy, and heatmaps in `audit/` before training.

In [ ]:
if NEED_T2TT:
    run(TEACHER_STAGE, 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest', MANIFEST,
        '--mt-attention', T2TT_ATTENTION_ROOT, '--alignment', CTC_ROOT, '--units', Path(ROOT) / 'units',
        '--config', BACKBONE_CONFIG, '--output', TEACHER_ROOT, '--resume')
    run(TEACHER_AUDIT_STAGE, 'alignment_transfer_s2ut.diagnose_alignment', '--manifest', MANIFEST,
        '--teacher', TEACHER_ROOT, '--units', Path(ROOT) / 'units', '--config', BACKBONE_CONFIG,
        '--split', 'dev', '--max-samples', 500, '--heatmaps', 20, '--output', TEACHER_AUDIT_ROOT, '--overwrite', *LAYER_FLAGS)
else:
    log_event('teacher-preparation', 'skipped', LOGS, 'teacher stage not requested')


## 7. Teacher-only top-p diagnostics and calibration

Use 16 logical training batches in fixed order and the same initialization. Review top-p and coverage before setting `MASK_TOP_P` and rerunning the settings cell. Calibration performs zero optimizer updates. The median estimators target a worst-to-median ratio of 3 for τ separately for KL and Mask, and a cross-attention gradient ratio of 0.5 for λ.


In [ ]:
CALIBRATION_INPUTS = ['--baseline', str(BASELINE), '--manifest', str(MANIFEST),
    '--units', str(Path(ROOT) / 'units'), '--teacher', str(TEACHER_ROOT), '--batches', str(CALIBRATION_BATCHES)]
topp_saved = any((Path(PERSISTENT) / 'stages' / ('research-v3-top-p' + SCOPE_SUFFIX)).glob('*/snapshot.json'))
calibration_saved = any((Path(PERSISTENT) / 'stages' / ('research-v3-calibration' + SCOPE_SUFFIX)).glob('*/snapshot.json'))
if RUN_MASK_TOPP_DIAGNOSTICS or topp_saved:
    run('research-v3-top-p' + SCOPE_SUFFIX, 'alignment_transfer_s2ut.alignment_calibration', 'top-p',
        *CALIBRATION_INPUTS, '--output', TOPP_REPORT, gpu=True)
else:
    log_event('top-p-diagnostics', 'skipped', LOGS, 'RUN_MASK_TOPP_DIAGNOSTICS=False')
if TOPP_REPORT.is_file():
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display, Image
    top_p_report = json.loads(TOPP_REPORT.read_text(encoding='utf-8'))
    display(pd.DataFrame([{'p':p, **{k:v for k,v in values.items() if k not in ('by_layer','by_source_length','histogram')}}
        for p,values in top_p_report['candidates'].items()]))
    for p,values in top_p_report['candidates'].items():
        print('p=', p, 'layer/source-length coverage')
        display(pd.DataFrame(values['by_layer']).T)
        display(pd.DataFrame(values['by_source_length']).T)
        plt.plot([(i+.5)/10 for i in range(10)], values['histogram'], label=p)
    plt.xlabel('Mask coverage'); plt.ylabel('Valid teacher rows'); plt.legend()
    plt.savefig(TOPP_ROOT / 'coverage.png'); plt.close()
    display(Image(filename=str(TOPP_ROOT / 'coverage.png')))
if RUN_ALIGNMENT_CALIBRATION or calibration_saved:
    if MASK_TOP_P not in (.8,.9,.95):
        raise ValueError('Review the top-p diagnostics and set MASK_TOP_P to 0.8, 0.9, or 0.95.')
    if not TOPP_REPORT.is_file():
        raise ValueError('Run or restore the top-p diagnostics before calibration.')
    run('research-v3-calibration' + SCOPE_SUFFIX, 'alignment_transfer_s2ut.alignment_calibration', 'calibrate',
        *CALIBRATION_INPUTS, '--diagnostics', TOPP_REPORT, '--mask-top-p', MASK_TOP_P,
        '--output', CALIBRATION, gpu=True)
else:
    log_event('alignment-calibration', 'skipped', LOGS, 'RUN_ALIGNMENT_CALIBRATION=False')
if CALIBRATION.is_file():
    calibrated = json.loads(CALIBRATION.read_text(encoding='utf-8'))
    print({key:calibrated[key] for key in ('lambda_kl','lambda_mask','tau_kl','tau_mask','selected_top_p','optimizer_updates')})


In [ ]:
# All three conditions use the same calibration, decoder layers, and Baseline recipe.
for method, label, enabled in RESEARCH_CONDITIONS:
    if not enabled:
        log_event(method, 'skipped', LOGS, 'training switch is off')
        continue
    if not CALIBRATION.is_file() or MASK_TOP_P not in (.8,.9,.95):
        raise ValueError('Run the top-p diagnostics, set MASK_TOP_P explicitly, and complete calibration.')
    folder = RESEARCH_ROOT / method
    arguments = ['--baseline', str(BASELINE), '--output', str(folder), '--method', method,
        '--manifest', str(MANIFEST), '--units', str(Path(ROOT) / 'units'), '--teacher', str(TEACHER_ROOT),
        '--calibration', str(CALIBRATION), '--mask-top-p', str(MASK_TOP_P), '--all-layers', '--resume']
    plan = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.benchmark', 'plan', *arguments, name='plan-' + method)
    print({'condition':label, 'training_budget':plan['training_budget'], 'initialization':plan['initialization'],
        'selection':plan['selection_policy'], 'calibration_hash':plan['transfer']['calibration_hash']})
    run('research-v3-' + method + SCOPE_SUFFIX, 'alignment_transfer_s2ut.benchmark', 'train', *arguments, gpu=True)


## 8. Shared inference and dev evaluation

Use the teacher-free native fairseq generator with beam 10, `max_len_a=1`, and `max_len_b=200` for every condition. Compare after all selected best checkpoints are available.

Lower UER and higher ASR-BLEU are better. The table shows an em dash when ASR is skipped. Reuse completed T2TT models and save comparison JSON per combination.

In [ ]:
conditions = []
from IPython.display import Markdown, display

if not GENERATE_WAVEFORMS:
    log_event('waveform-generation', 'skipped', LOGS, 'GENERATE_WAVEFORMS=False')
if not ASR_BLEU:
    log_event('asr-bleu', 'skipped', LOGS, 'ASR_BLEU=False')

transfer_conditions = []
for method, label, enabled in RESEARCH_CONDITIONS:
    folder = RESEARCH_ROOT / method
    if enabled or (folder / 'checkpoints/checkpoint_best.pt').is_file():
        transfer_conditions.append((method, label, folder, RESEARCH_ROOT / 'registrations' / (method + '.json'), 'research-v3-register-' + method))
missing = [label for _, label, folder, _, _ in transfer_conditions
           if not (folder / 'checkpoints/checkpoint_best.pt').is_file()]
if missing:
    log_event('dev-comparison', 'skipped', LOGS, 'Waiting for training to complete: ' + ', '.join(missing))
elif not transfer_conditions:
    log_event('dev-comparison', 'skipped', LOGS, 'enable and complete a research condition first')
else:
    conditions = [('baseline', 'Baseline', BASELINE)]
    for name, label, folder, record, registration in transfer_conditions:
        run(registration, 'alignment_transfer_s2ut.benchmark', 'register-transfer', '--run', folder,
            '--checkpoint', folder / 'checkpoints/checkpoint_best.pt', '--output', record, '--resume')
        conditions.append((name, label, record))
    records = [record for _, _, record in conditions]
    comparison_name = '-'.join(name for name, _, _ in conditions)
    COMPARISON_DIR = Path(ROOT) / 'comparison' / comparison_name
    run(comparison_name + '-compare-training', 'alignment_transfer_s2ut.benchmark', 'compare', *records,
        '--output', COMPARISON_DIR / 'training-comparison.json', '--resume')
    source_only = Path(ROOT) / 'dev-source/dev-source.jsonl'
    run('dev-source', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest', MANIFEST, '--split', 'dev', '--output', source_only, '--resume')
    if GENERATE_WAVEFORMS:
        run('vocoder', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml')
    evaluations = []
    table = ['| Condition | ASR-BLEU | UER |', '|---|---:|---:|']
    for name, label, record in conditions:
        predictions = Path(ROOT) / 'evaluation' / name / 'dev-predictions.jsonl'
        vocoder = ['--vocoder-config', 'configs/s2ut/vocoder.yaml'] if GENERATE_WAVEFORMS else ['--units-only']
        run(name + '-infer', 'alignment_transfer_s2ut.benchmark', 'infer', '--record', record, '--input', source_only,
            '--output', predictions, '--device', 'cuda', '--resume', *vocoder, gpu=True)
        metrics = predictions.with_name('dev-metrics.json')
        asr = ['--asr-config', 'configs/experiments/evaluation.yaml', '--device', 'cuda'] if ASR_BLEU else []
        run(name + '-evaluate', 'alignment_transfer_s2ut.evaluate', '--manifest', MANIFEST, '--split', 'dev',
            '--units', Path(ROOT) / 'units', '--predictions', predictions, '--output', metrics, '--resume', *asr, gpu=ASR_BLEU)
        evaluations.append(metrics)
        values = json.loads(metrics.read_text())
        bleu = f"{values['asr_bleu']:.2f}" if values.get('asr_bleu') is not None else '—'
        uer = f"{values['unit_error_rate']:.4f}" if values.get('unit_error_rate') is not None else '—'
        table.append(f'| {label} | {bleu} | {uer} |')
    run(comparison_name + '-comparison', 'alignment_transfer_s2ut.benchmark', 'compare', *records, '--evaluations', *evaluations,
        '--output', COMPARISON_DIR / 'comparison.json', '--resume')
    print('Comparison report:', COMPARISON_DIR / 'comparison.json')
    display(Markdown('\n'.join(table)))


In [ ]:
# Run teacher-forced dev diagnostics separately from generation.
if globals().get('conditions') and CALIBRATION.is_file():
    import pandas as pd
    from IPython.display import display, Image
    completed = []
    for name, label, record in conditions:
        output = RESEARCH_ROOT / 'alignment-diagnostics' / name / 'diagnostics.json'
        run('research-v3-alignment-' + name + SCOPE_SUFFIX, 'alignment_transfer_s2ut.alignment_report', 'diagnose',
            '--baseline', BASELINE, '--record', record, '--manifest', MANIFEST, '--units', Path(ROOT) / 'units',
            '--teacher', TEACHER_ROOT, '--calibration', CALIBRATION, '--samples', 32, '--output', output, gpu=True)
        result = json.loads(output.read_text(encoding='utf-8'))
        print(label, 'KL mean/worst:', result['kl_mean'], result['kl_worst'])
        display(pd.DataFrame({'KL':result['per_layer_kl'], 'Mask loss':result['per_layer_mask_loss'],
            'Region mass':result['teacher_region_mass'], 'Worst-layer weight':result['worst_layer_weight_distribution']}))
        if name != 'baseline':
            dev_pair = next(json.loads(line)['pair_id'] for line in MANIFEST.read_text(encoding='utf-8').splitlines()
                if line.strip() and json.loads(line)['split']=='dev')
            visual = RESEARCH_ROOT / 'attention-visualization' / name
            run('research-v3-visualize-' + name + SCOPE_SUFFIX, 'alignment_transfer_s2ut.visualize_attention',
                '--manifest', MANIFEST, '--checkpoint', RESEARCH_ROOT / name / 'checkpoints/checkpoint_best.pt',
                '--units', Path(ROOT) / 'units', '--teacher', TEACHER_ROOT, '--pair-id', dev_pair,
                '--device', 'cuda', '--output', visual, '--overwrite', gpu=True)
            for path in visual.glob('*.png'):
                display(Image(filename=str(path)))
        if name != 'baseline':
            completed.append(RESEARCH_ROOT / name)
    if completed:
        curves_root = RESEARCH_ROOT / 'curves'
        run('research-v3-curves' + SCOPE_SUFFIX, 'alignment_transfer_s2ut.alignment_report', 'curves',
            '--runs', *completed, '--output', curves_root)
        for folder in completed:
            display(Image(filename=str(curves_root / (folder.name + '.png'))))
else:
    log_event('layer-diagnostics', 'skipped', LOGS, 'complete calibration and common evaluation first')


## Storage and recovery

Write frequently to the VM and publish verified benchmark ZIPs to Drive. Rerun with the original revision, settings, `ROOT`, and `PERSISTENT` to restore optimizer, RNG, and iterator from the last verified snapshot. The calibration artifact must match on resume.

Use a separate research output for a different method, top-p, teacher, or calibration. The main three transfer conditions share one calibration artifact. [Method, microbatch definition, and calibration](../../docs/METHOD.md)
